In [2]:
import numpy as np
import pandas as pd

In [42]:
y = pd.read_csv('../data/raw/ivs_spx_16_26.csv')
y['Dates'] = y['Dates'].astype('datetime64[s]')
y = y.set_index('Dates')

y.head()

,30DAY_IMPVOL_80%MNY_DF,30DAY_IMPVOL_90.0%MNY_DF,30DAY_IMPVOL_95.0%MNY_DF,30DAY_IMPVOL_97.5%MNY_DF,30DAY_IMPVOL_100.0%MNY_DF,30DAY_IMPVOL_102.5%MNY_DF,30DAY_IMPVOL_105.0%MNY_DF,30DAY_IMPVOL_110.0%MNY_DF,30DAY_IMPVOL_120%MNY_DF,60DAY_IMPVOL_80%MNY_DF,...,18MTH_IMPVOL_120%MNY_DF,24MTH_IMPVOL_80%MNY_DF,24MTH_IMPVOL_90.0%MNY_DF,24MTH_IMPVOL_95.0%MNY_DF,24MTH_IMPVOL_97.5%MNY_DF,24MTH_IMPVOL_100.0%MNY_DF,24MTH_IMPVOL_102.5%MNY_DF,24MTH_IMPVOL_105.0%MNY_DF,24MTH_IMPVOL_110.0%MNY_DF,24MTH_IMPVOL_120%MNY_DF
Dates,,,,,,,,,,,,,,,,,,,,,
2016-04-01,31.4422,20.4818,15.7749,13.5518,11.1926,8.7991,8.6680,12.3428,13.0078,26.5925,...,11.8507,22.6374,20.1359,18.9134,18.3122,17.6810,17.0812,16.4947,15.3507,13.3202
2016-04-04,32.3816,21.0527,16.4633,14.4232,12.1854,9.7695,9.1445,12.4163,13.2117,27.3005,...,12.0997,22.8203,20.3300,19.1046,18.4937,17.8915,17.2888,16.7016,15.5572,13.5400
2016-04-05,32.8637,22.0677,17.6430,15.6324,13.4519,11.2138,9.7895,12.3206,12.3206,28.1957,...,12.5537,23.2521,20.7511,19.5040,18.8611,18.3484,17.7543,17.1416,15.9772,13.9185
2016-04-06,32.3115,20.9377,16.4275,14.4957,12.2162,9.9239,8.8714,11.5959,11.5959,27.3145,...,12.2230,22.9935,20.4688,19.2367,18.6073,18.0355,17.4445,16.8486,15.6908,13.6485
2016-04-07,33.5882,22.9630,18.5421,16.4382,14.1363,11.4923,9.7613,11.6225,11.7970,28.9461,...,12.8713,23.5757,21.0630,19.8167,19.1972,18.6367,18.0572,17.4823,16.3173,14.2314


## Split data

In [40]:
n = len(y)
y_train = y.iloc[:int(n*0.8)].to_numpy().astype(float)
y_test = y.iloc[int(n*0.8):].to_numpy().astype(float)

print(f"Total: {n}, train: {len(y_train)}, test: {len(y_test)}")

Total: 2647, train: 2117, test: 530


## VAR

In [ ]:
from statsmodels.tsa.api import VAR
from datetime import datetime
from collections import defaultdict

horizons = [1, 14, 30, 90, 180, 365]

class VAR_Results:
    def __init__(self, res):
        self.res = res
        self.p = res.k_ar
        self.forecasts = defaultdict(list)
        self.errors = defaultdict(list)
        self.mse = {}

    def compute_error(self, h: int, d: datetime, y: pd.DataFrame) -> None:
        """Compute forecasting error.
        
        Args:
            h: forecast horizon
            d: date to forecast
            y: dataset to backtest

        Return:
            Forecast error for the chosen date
        """
        d = y.index.get_loc(d)
        actual = y.iloc[d]
        inputs = y.iloc[d-h-self.p+1:d-h+1].to_numpy()
        pred = self.res.forecast(inputs, steps=h)[-1]
        self.forecasts[h].append(pred)
        self.errors[h].append((actual - pred)**2)
        
    def compute_mse(self, h) -> float:
        """Compute mean squared error from current sum of squared errors.

        Args:
            h: forecast horizon
        
        Return:
            Mean squared error
        """
        try:             
            self.mse[h] = np.mean(self.errors[h])
            return self.mse[h]
        
        except KeyError:
            print("There are no predictions for this forecast horizon.")

var = VAR(y_train)

print("Fitting VAR(1)...")
res_var_1 = VAR_Results(var.fit(1))
print("Successfully fit VAR(1)")

var_models = [res_var_1, res_var_aic, res_var_bic]

for h in horizons:
    for d in y.index[int(n*0.8):]:
        for model in var_models:
            model.compute_error(h, d, y)

    for model in var_models:
            model.compute_mse(h)

Fitting VAR(1)...
Successfully fit VAR(1)


In [ ]:
res_var_1.fit()

In [54]:
var = VAR(y_train)

print("Fitting VAR(1)...")
res_var_1 = VAR_Results(var.fit(1))

# print("Fitting VAR(p) with AIC...")
# res_var_aic = VAR_Results(var.fit(maxlags=20, ic='aic'))

# print("Fitting VAR(p) with BIC...")
# res_var_bic = VAR_Results(var.fit(maxlags=20, ic='bic'))

# print(f"Lag order selected by AIC: {res_var_aic.p}")
# print(f"Lag order selected by BIC: {res_var_bic.p}")

Fitting VAR(1)...


In [58]:
var_models = [res_var_1]#, res_var_aic, res_var_bic]

for h in horizons:
    for d in y.index[int(n*0.8):]:
        for model in var_models:
            model.compute_error(h, d, y)

    for model in var_models:
            model.compute_mse(h)

KeyError: -1